In [1]:
import os
from dotenv import load_dotenv
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from datetime import datetime
from connector import get_db_connection

In [2]:
conn = get_db_connection()
query = """
SELECT date_taken, date_taken_granularity
FROM photo 
WHERE date_taken IS NOT NULL
"""
df = pd.read_sql_query(query, conn)
conn.close()


/tmp/ipykernel_34944/1382970712.py:7: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql_query(query, conn)


In [3]:
df['year'] = pd.to_datetime(df['date_taken']).dt.year
df = df.dropna(subset=['year'])
df = df[(df['year'] >= 1800) & (df['year'] <= 2026)]
print(f"Loaded {len(df)} photos spanning {df['year'].min()}–{df['year'].max()}")

Loaded 44985 photos spanning 1825–2025


In [ ]:
def plot_year_and_granularity(df):
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('Photo Count per Year', 'Granularity (Mean ± Std per Year)'),
        vertical_spacing=0.2,
        row_heights=[0.7, 0.3]
    )
    year_counts = df['year'].value_counts().sort_index()
    fig.add_trace(
        go.Bar(
            x=year_counts.index,
            y=year_counts.values,
            name='Photo Count',
            marker_color='steelblue',
            opacity=0.8
        ),
        row=1, col=1
    )
    year_stats = df.groupby('year')['date_taken_granularity'].agg(['mean', 'std', 'count']).reset_index()
    year_stats['error'] = year_stats['std'] / np.sqrt(year_stats['count'])  # Standard error

    fig.add_trace(
        go.Bar(
            x=year_stats['year'],
            y=year_stats['mean'],
            name='Granularity Mean',
            marker_color='darkorange',
            opacity=0.8,
            error_y=dict(
                type='data',
                array=year_stats['error'],
                visible=True,
                color='black',
                thickness=1,
                width=4
            )
        ),
        row=2, col=1
    )

    fig.update_layout(
        height=700,
        showlegend=False,
        title_text="Photo Date Distribution & Granularity Analysis",
        title_font_size=20,
        xaxis_title="Year",
        yaxis_title="Number of Photos",
        xaxis2_title="Year",
        yaxis2_title="Granularity (Mean ± SEM)"
    )
    fig.update_xaxes(
        tickmode='linear',
        tick0=df['year'].min() // 10 * 10,
        dtick=10,
        row=1, col=1
    )
    fig.update_xaxes(
        tickmode='linear',
        tick0=df['year'].min() // 10 * 10,
        dtick=10,
        row=2, col=1
    )
    fig.show()
    print("\nSummary:")
    print(f"Total photos: {len(df)}")
    print(f"Years covered: {df['year'].min()}–{df['year'].max()}")
    print(f"Peak year: {year_counts.idxmax()} ({year_counts.max()} photos)")
    print(f"Avg photos/year: {len(df)/len(year_counts):.1f}")
    print(f"Avg granularity: {df['date_taken_granularity'].mean():.2f} ± {df['date_taken_granularity'].std():.2f}")

In [15]:
plot_year_and_granularity(df)


Summary:
Total photos: 44985
Years covered: 1825–2025
Peak year: 1915 (10578 photos)
Avg photos/year: 240.6
Avg granularity: 7.19 ± 1.30


In [6]:
def sample_by_year(df, max_per_year=500):
    """For each year, sort by granularity ASCENDING (keep lowest), take top N"""
    sampled_dfs = []   
    for year, group in df.groupby('year'):
        if len(group) > max_per_year:
            sampled = group.sort_values('date_taken_granularity').head(max_per_year)
            sampled_dfs.append(sampled)
        else:
            sampled_dfs.append(group)
    return pd.concat(sampled_dfs, ignore_index=True)

df500 = sample_by_year(df, max_per_year=500)
print(f"Kept {len(df500)} photos spanning {df500['year'].min()}–{df500['year'].max()}")

Kept 17611 photos spanning 1825–2025


In [7]:
plot_year_and_granularity(df500)


Summary:
Total photos: 17611
Years covered: 1825–2025
Peak year: 1861 (500 photos)
Avg photos/year: 94.2
Avg granularity: 6.20 ± 1.49
